<a href="https://colab.research.google.com/github/ailingomezromay/Grupo-4--TP-BigData-Burnout/blob/rama-vicky/TP_Grupo%204_BigData_Burnout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP Final - Big Data y MLOps

Predicción de riesgo de burnout en empleados del sector tecnológico

# Grupo 4
Ailin Gomez Romay  
Carlos Lazzarino  
Victoria Reppucci  
Vanessa Vidal

#1. Introducción

# Objetivo del trabajo

El objetivo de este trabajo es desarrollar un modelo de Machine Learning capaz de predecir el riesgo de burnout en empleados del sector tecnológico, a partir de variables demográficas, laborales y relacionadas con la salud mental.

#Dataset

El dataset fue obtenido de Kaggle ( https://www.kaggle.com/datasets/suhanigupta04/employee-mental-health-and-burnout-dataset ).

Contiene aproximadamente 150.000 registros sintéticos de empleados del sector tecnológico, con 25 variables que incluyen información demográfica, condiciones laborales e indicadores de salud mental.

El volumen del dataset justifica el uso de PySpark como framework de procesamiento distribuido. Además, las variables disponibles tienen una interpretación intuitiva y relevante para el problema: horas trabajadas, nivel de estrés, calidad de sueño, entre otras.

#Problema a resolver

Se busca identificar qué empleados presentan mayor riesgo de burnout a partir de variables observables. Este tipo de análisis puede resultar de utilidad para áreas de Recursos Humanos y bienestar organizacional, ya que permite anticipar situaciones de riesgo y tomar decisiones preventivas de manera informada.

# 2. Librerías e imports

In [ ]:
# Instalaciones necesarias
!pip install optuna mlflow shap fastapi uvicorn pydantic --quiet

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import optuna

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression as SklearnLR

import mlflow
import mlflow.spark
from mlflow.tracking import MlflowClient
import shap

# Suprimir logs verbose de Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print('Librerías cargadas correctamente.')

# 3. Carga del dataset

En esta sección se carga el dataset seleccionado utilizando la funcionalidad de subida de archivos de Google Colab.

Una vez cargado, se realiza una inspección inicial para verificar la cantidad de registros y variables disponibles.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
data_path = 'tech_mental_health_burnout.csv'
df = pd.read_csv(data_path)

print(f'Dimensiones del dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas')
df.head()

In [ ]:

df.info()

#4.Exploración inicial del dataset (EDA)

El dataset contiene información sobre empleados del sector tecnológico, incluyendo variables demográficas, laborales y de salud mental. A partir de una primera inspección, se observa que la mayoría de las variables son numéricas, aunque también hay variables categóricas que deberán ser transformadas para el modelado.

# 4.1. Análisis de la variable objetivo

Se analiza la distribución de burnout_level para identificar posibles desbalances entre clases.



In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='burnout_level', order=df['burnout_level'].value_counts().index)
plt.title('Distribución de burnout_level')
plt.xlabel('Nivel de burnout')
plt.ylabel('Cantidad de empleados')
plt.tight_layout()
plt.show()

print(df['burnout_level'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

Se observa un fuerte desbalance de clases: la categoría Low concentra aproximadamente el 88% de los registros, mientras que High representa una fracción muy pequeña.

Este desbalance puede afectar negativamente el desempeño del modelo, ya que los algoritmos tienden a favorecer la clase mayoritaria. Por este motivo, se decide reformular el problema como clasificación binaria, agrupando Moderate y High en una única clase de riesgo (burnout_risk = 1).

# 4.2  Relación con variables clave y burnout



Relación entre estrés y burnout

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="burnout_level", y="stress_level")
plt.title("Estrés según nivel de burnout")
plt.show()

Estrés: presenta la separación más pronunciada entre grupos, lo que sugiere que es el predictor individual más relevante.

Relación entre horas trabajadas y burnout

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="burnout_level", y="work_hours_per_week")
plt.title("Horas trabajadas según burnout")
plt.show()

Horas trabajadas: la tendencia es consistente (más horas, mayor burnout), aunque la diferencia entre grupos no es tan marcada.

Relación entre sueño y burnout

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="burnout_level", y="sleep_hours")
plt.title("Horas de sueño según burnout")
plt.show()

Horas de sueño: la diferencia entre Low y High es menor de lo esperado, aunque la dirección de la relación se mantiene.

# 4.3. Correlaciones

In [ ]:
cols_corr = [
    'work_hours_per_week', 'overtime_hours', 'job_satisfaction',
    'manager_support', 'work_life_balance', 'sleep_hours',
    'stress_level', 'anxiety_score', 'depression_score', 'burnout_score'
]

corr = df[cols_corr].corr()

plt.figure(figsize=(10, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, square=True)
plt.title('Matriz de correlación — variables numéricas')
plt.tight_layout()
plt.show()



Las variables de salud mental (stress_level, anxiety_score, depression_score) presentan alta correlación positiva con burnout_score. Variables como work_life_balance y sleep_hours muestran correlación negativa, comportándose como factores protectores.


#4.4 Cardinalidad de variables categóricas y valores nulos

In [ ]:
print(f'Total de valores nulos: {df.isnull().sum().sum()}\n')

for col in ['gender', 'job_role', 'company_size', 'work_mode']:
    print(f'{col}: {df[col].nunique()} categorías — {df[col].unique().tolist()}')

## 5. Modelado con PySpark

En esta sección se construye el pipeline de procesamiento y se entrenan los modelos de clasificación base. El uso de pipelines garantiza la reproducibilidad y facilita la integración con MLflow.

#5.1 Inicialización de Spark

In [ ]:
spark = SparkSession.builder \
    .appName('TP_BigData_Burnout') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print(spark)

#5.2 Preparación de la variable objetivo

Se convierte el DataFrame de Pandas a Spark y se crea la variable binaria burnout_risk. Luego se eliminan burnout_level y burnout_score para evitar data leakage.

In [ ]:
df_spark = spark.createDataFrame(df)

df_spark = df_spark.withColumn(
    'burnout_risk',
    F.when(F.col('burnout_level').isin('Moderate', 'High'), 1).otherwise(0)
)

df_spark = df_spark.drop('burnout_level', 'burnout_score')
df_spark.printSchema()

#5.3 Pipeline de preprocesamiento

Limitación conocida: el StringIndexer se ajusta sobre el dataset completo antes del split, lo que podría introducir data leakage leve en categorías poco frecuentes. En un entorno productivo, el ajuste debería realizarse exclusivamente sobre el conjunto de entrenamiento.

In [ ]:
categorical_cols = [
    "gender", "job_role", "company_size", "work_mode"
]

numeric_cols = [
    "age", "experience_years", "work_hours_per_week", "overtime_hours",
    "meetings_per_day", "deadlines_missed", "job_satisfaction",
    "manager_support", "work_life_balance", "sleep_hours",
    "physical_activity_days", "screen_time_hours", "caffeine_intake",
    "social_support_score", "has_therapy", "stress_level",
    "anxiety_score", "depression_score", "seeks_professional_help"
]

target_col = "burnout_risk"

indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep")
    for col in categorical_cols
]

encoders = [
    OneHotEncoder(inputCol=f"{col}_idx", outputCol=f"{col}_ohe")
    for col in categorical_cols
]

feature_cols = numeric_cols + [f"{col}_ohe" for col in categorical_cols]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

train_df, test_df = df_spark.randomSplit([0.8, 0.2], seed=42)

print("Train:", train_df.count())
print("Test:", test_df.count())

#5.4 Entrenamiento de modelos base

In [ ]:
# Regresión Logística
lr = LogisticRegression(featuresCol='features', labelCol='burnout_risk')
pipeline_lr = Pipeline(stages=indexers + encoders + [assembler, lr])
model_lr    = pipeline_lr.fit(train_df)
preds_lr    = model_lr.transform(test_df)

# Random Forest
rf = RandomForestClassifier(featuresCol='features', labelCol='burnout_risk',
                             numTrees=50, seed=42)
pipeline_rf = Pipeline(stages=indexers + encoders + [assembler, rf])
model_rf    = pipeline_rf.fit(train_df)
preds_rf    = model_rf.transform(test_df)

#5.5 Evaluación de modelos base

In [ ]:
evaluator_auc = BinaryClassificationEvaluator(
    labelCol='burnout_risk', metricName='areaUnderROC')
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol='burnout_risk', metricName='f1')

auc_lr = evaluator_auc.evaluate(preds_lr)
f1_lr  = evaluator_f1.evaluate(preds_lr)
auc_rf = evaluator_auc.evaluate(preds_rf)
f1_rf  = evaluator_f1.evaluate(preds_rf)

print(f"{'Modelo':<25} {'AUC':>8} {'F1':>8}")
print('-' * 43)
print(f"{'Logistic Regression':<25} {auc_lr:>8.4f} {f1_lr:>8.4f}")
print(f"{'Random Forest':<25} {auc_rf:>8.4f} {f1_rf:>8.4f}")

La Regresión Logística obtuvo mejor AUC y F1. Se selecciona como modelo candidato por su mejor desempeño y mayor interpretabilidad. Estos resultados serán refinados mediante búsqueda de hiperparámetros en la sección siguiente.

#5.6 Matriz de confusión

Para complementar el AUC calculamos el F1-score (0.93) y la matriz de confusión. Esto es especialmente importante dado el desbalance: el F1 penaliza los falsos negativos, que en este contexto son los casos de burnout que el modelo no detecta.

In [ ]:
preds_pd = preds_lr.select('burnout_risk', 'prediction').toPandas()
cm       = confusion_matrix(preds_pd['burnout_risk'], preds_pd['prediction'])
disp     = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['Sin riesgo', 'En riesgo'])

fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False)
ax.set_title('Matriz de confusión — Regresión Logística (baseline)')
plt.tight_layout()
plt.show()

#5.7 Interpretabilidad con SHAP

Se utiliza SHAP con una Regresión Logística equivalente entrenada en scikit-learn sobre las variables numéricas, para analizar el impacto de cada variable en las predicciones.




In [ ]:
train_pd = train_df.select(numeric_cols + ['burnout_risk']).toPandas().dropna()
X_train  = train_pd[numeric_cols]
y_train  = train_pd['burnout_risk']

sk_model = SklearnLR(max_iter=1000)
sk_model.fit(X_train, y_train)

explainer   = shap.LinearExplainer(sk_model, X_train)
shap_values = explainer.shap_values(X_train)

shap.summary_plot(shap_values, X_train, plot_type='bar',
                  title='Importancia de variables — SHAP (Regresión Logística)')

Las variables con mayor impacto son stress_level, work_hours_per_week y work_life_balance, lo cual es consistente con el análisis exploratorio.

#6. MLflow

In [ ]:
mlflow.set_experiment("TP_BigData_Burnout")

Registrar Logistic

In [ ]:

mlflow.end_run()

with mlflow.start_run(run_name="Logistic_Regression") as run_lr:
    mlflow.log_param("model", "Logistic Regression")
    mlflow.log_param("train_size", train_df.count())
    mlflow.log_metric("AUC", auc_lr)
    mlflow.log_metric("F1", f1_lr)


    mlflow.spark.log_model(model_lr, "model")
    run_id_lr = run_lr.info.run_id


model_uri = f"runs:/{run_id_lr}/model"
mlflow.register_model(model_uri, "Burnout_Model")

Registrar Random Forest

In [ ]:
with mlflow.start_run(run_name="Random_Forest"):
    mlflow.log_param("model", "Random Forest")
    mlflow.log_param("numTrees", 50)
    mlflow.log_param("train_size", train_df.count())
    mlflow.log_metric("AUC", auc_rf)
    mlflow.log_metric("F1", f1_rf)

## Registro de experimentos con MLflow

Usamos MLflow para registrar los dos modelos y poder comparar métricas desde la UI. Además de AUC logueamos también F1 y los parámetros principales de cada modelo.

#7.Búsqueda de hiperparámetros con Optuna + MLflow Tracking

En esta sección se optimizan los dos modelos elegidos del trabajo:
- **Logistic Regression**
- **Random Forest**

Para cada trial de Optuna se registran:
- hiperparámetros probados
- métricas de desempeño (**AUC** y **F1**)
- nombre del modelo

Todo queda guardado en **MLflow Tracking**, así que después se puede comparar fácilmente desde la UI.


In [ ]:
# Experimento separado para mantener los runs de HPO
# independientes de los modelos base
mlflow.set_experiment('TP_BigData_Burnout_HPO')
mlflow.end_run()

print('Experimento MLflow configurado: TP_BigData_Burnout_HPO')

### 7.2 Optimización — Regresión Logística

| Hiperparámetro | Descripción | Rango |
|---|---|---|
| `regParam` | Fuerza de regularización | [1e-4, 10] log-escala |
| `elasticNetParam` | Balance L1/L2 (0=Ridge, 1=Lasso) | [0.0, 1.0] |
| `maxIter` | Iteraciones máximas | {50, 100, ..., 300} |

In [ ]:
def objective_lr(trial):
    reg_param   = trial.suggest_float('regParam',         1e-4, 10.0, log=True)
    elastic_net = trial.suggest_float('elasticNetParam',  0.0,  1.0)
    max_iter    = trial.suggest_int('maxIter',            50,   300,  step=50)

    clf = LogisticRegression(
        featuresCol='features', labelCol='burnout_risk',
        regParam=reg_param, elasticNetParam=elastic_net, maxIter=max_iter
    )
    pipeline = Pipeline(stages=indexers + encoders + [assembler, clf])

    with mlflow.start_run(run_name=f'LR_trial_{trial.number:02d}', nested=True):
        mlflow.log_param('model',           'LogisticRegression')
        mlflow.log_param('regParam',        round(reg_param, 6))
        mlflow.log_param('elasticNetParam', round(elastic_net, 4))
        mlflow.log_param('maxIter',         max_iter)

        model = pipeline.fit(train_df)
        preds = model.transform(test_df)
        auc   = evaluator_auc.evaluate(preds)
        f1    = evaluator_f1.evaluate(preds)

        mlflow.log_metric('AUC', auc)
        mlflow.log_metric('F1',  f1)

    return auc


with mlflow.start_run(run_name='HPO_LogisticRegression'):
    study_lr = optuna.create_study(
        direction='maximize',
        study_name='LR_burnout',
        sampler=optuna.samplers.TPESampler(seed=42)
    )
    study_lr.optimize(objective_lr, n_trials=20, show_progress_bar=True)

    best_lr = study_lr.best_trial
    mlflow.log_metric('best_AUC', best_lr.value)
    for k, v in best_lr.params.items():
        mlflow.log_param(f'best_{k}', v)

print(f'\n Mejor trial LR #{best_lr.number} — AUC: {best_lr.value:.4f}')
print(f'   Parámetros: {best_lr.params}')

### 7.3 Optimización — Random Forest

| Hiperparámetro | Descripción | Rango |
|---|---|---|
| `numTrees` | Número de árboles | {50, 100, ..., 300} |
| `maxDepth` | Profundidad máxima | [3, 15] |
| `minInfoGain` | Ganancia mínima para split | [0.0, 0.1] |

In [ ]:
def objective_rf(trial):
    num_trees     = trial.suggest_int('numTrees',     50,  300, step=50)
    max_depth     = trial.suggest_int('maxDepth',      3,   15)
    min_info_gain = trial.suggest_float('minInfoGain', 0.0, 0.1)

    clf = RandomForestClassifier(
        featuresCol='features', labelCol='burnout_risk',
        numTrees=num_trees, maxDepth=max_depth,
        minInfoGain=min_info_gain, seed=42
    )
    pipeline = Pipeline(stages=indexers + encoders + [assembler, clf])

    with mlflow.start_run(run_name=f'RF_trial_{trial.number:02d}', nested=True):
        mlflow.log_param('model',       'RandomForest')
        mlflow.log_param('numTrees',    num_trees)
        mlflow.log_param('maxDepth',    max_depth)
        mlflow.log_param('minInfoGain', round(min_info_gain, 5))

        model = pipeline.fit(train_df)
        preds = model.transform(test_df)
        auc   = evaluator_auc.evaluate(preds)
        f1    = evaluator_f1.evaluate(preds)

        mlflow.log_metric('AUC', auc)
        mlflow.log_metric('F1',  f1)

    return auc


with mlflow.start_run(run_name='HPO_RandomForest'):
    study_rf = optuna.create_study(
        direction='maximize',
        study_name='RF_burnout',
        sampler=optuna.samplers.TPESampler(seed=42)
    )
    study_rf.optimize(objective_rf, n_trials=20, show_progress_bar=True)

    best_rf = study_rf.best_trial
    mlflow.log_metric('best_AUC', best_rf.value)
    for k, v in best_rf.params.items():
        mlflow.log_param(f'best_{k}', v)

print(f'\n Mejor trial RF #{best_rf.number} — AUC: {best_rf.value:.4f}')
print(f'   Parámetros: {best_rf.params}')

### 7.4 Selección del mejor modelo

In [ ]:
print('=== Resumen HPO ===')
print(f'  Logistic Regression — mejor AUC: {best_lr.value:.4f} (trial #{best_lr.number})')
print(f'  Random Forest       — mejor AUC: {best_rf.value:.4f} (trial #{best_rf.number})')

if best_lr.value >= best_rf.value:
    BEST_MODEL_NAME = 'LogisticRegression'
    BEST_PARAMS     = best_lr.params
    BEST_AUC        = best_lr.value
    BEST_STUDY      = study_lr
else:
    BEST_MODEL_NAME = 'RandomForest'
    BEST_PARAMS     = best_rf.params
    BEST_AUC        = best_rf.value
    BEST_STUDY      = study_rf

print(f'\n Modelo seleccionado: {BEST_MODEL_NAME} (AUC = {BEST_AUC:.4f})')
print(f'   Hiperparámetros: {BEST_PARAMS}')

### 7.5 Visualización de los experimentos de HPO

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Gráfico 1: Historial de optimización
ax = axes[0]
for study, label, color in [
    (study_lr, 'Logistic Regression', '#1f77b4'),
    (study_rf, 'Random Forest',       '#ff7f0e')
]:
    best_so_far, current_best = [], 0
    for t in study.trials:
        if t.value and t.value > current_best:
            current_best = t.value
        best_so_far.append(current_best)
    ax.plot(range(1, len(best_so_far) + 1), best_so_far,
            marker='o', markersize=4, label=label, color=color)
ax.set_title('Historial de optimización')
ax.set_xlabel('Trial')
ax.set_ylabel('Mejor AUC acumulado')
ax.legend()
ax.grid(True, alpha=0.3)

# Gráfico 2: Importancia de hiperparámetros
ax = axes[1]
try:
    importances = optuna.importance.get_param_importances(BEST_STUDY)
    bars = ax.barh(list(importances.keys()), list(importances.values()), color='#2ca02c')
    ax.bar_label(bars, fmt='%.3f', padding=3)
    ax.set_title(f'Importancia de hiperparámetros\n({BEST_MODEL_NAME})')
    ax.set_xlabel('Importancia relativa')
    ax.grid(True, axis='x', alpha=0.3)
except Exception as e:
    ax.text(0.5, 0.5, f'No disponible\n{e}', ha='center', va='center', transform=ax.transAxes)

# Gráfico 3: Distribución de AUC por modelo
ax = axes[2]
auc_lr_vals = [t.value for t in study_lr.trials if t.value is not None]
auc_rf_vals = [t.value for t in study_rf.trials if t.value is not None]
ax.boxplot([auc_lr_vals, auc_rf_vals],
           labels=['Logistic\nRegression', 'Random\nForest'],
           patch_artist=True,
           boxprops=dict(facecolor='#aec7e8'),
           medianprops=dict(color='#d62728', linewidth=2))
ax.set_title('Distribución de AUC por modelo\n(todos los trials)')
ax.set_ylabel('AUC')
ax.grid(True, axis='y', alpha=0.3)

plt.suptitle('Análisis de búsqueda de hiperparámetros — Optuna + MLflow',
             fontsize=13, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

7.6 Tabla resumen de trials

In [ ]:
def study_to_df(study, model_name):
    rows = []
    for t in study.trials:
        if t.value is not None:
            row = {'modelo': model_name, 'trial': t.number, 'AUC': round(t.value, 4)}
            row.update({k: round(v, 5) if isinstance(v, float) else v
                        for k, v in t.params.items()})
            rows.append(row)
    return pd.DataFrame(rows)

df_all_trials = pd.concat(
    [study_to_df(study_lr, 'LogisticRegression'),
     study_to_df(study_rf, 'RandomForest')],
    ignore_index=True
)

print('Top 10 trials por AUC:')
display(df_all_trials.sort_values('AUC', ascending=False).head(10))

---
## 8. Model Registry

Con los mejores hiperparámetros identificados por Optuna, se reentrena el modelo final y se registra en el **MLflow Model Registry** para su versionado y posterior despliegue.

#9. API REST para consumo del modelo


## 10. Conclusiones

En este trabajo se desarrolló un pipeline completo de Machine Learning para la predicción de riesgo de burnout en empleados del sector tecnológico, utilizando PySpark como framework de procesamiento distribuido y MLflow para la gestión de experimentos y versionado de modelos.

### Principales hallazgos

- El análisis exploratorio identificó al **nivel de estrés** como el predictor más relevante, seguido por las horas trabajadas y el balance vida-trabajo. Estos resultados fueron validados posteriormente por el análisis SHAP.
- Se entrenaron y compararon dos modelos. La **Regresión Logística** fue seleccionada como modelo final por su mejor desempeño (AUC y F1) y mayor interpretabilidad.
- La búsqueda de hiperparámetros con **Optuna** (20 trials por modelo, algoritmo TPE) permitió mejorar las métricas respecto a los modelos base, con todos los experimentos trazados en MLflow.
- El modelo final fue registrado en el **MLflow Model Registry**

### Limitaciones y trabajo futuro

- Aplicar técnicas de balanceo de clases (SMOTE, `class_weight`) para mejorar la detección de la clase minoritaria.
- Optimizar el umbral de decisión más allá del valor por defecto de 0.5, priorizando la reducción de falsos negativos.
- Ajustar el `StringIndexer` exclusivamente sobre el conjunto de entrenamiento para eliminar el riesgo de data leakage.
